# ECE501 Week 2 — Digital Image Processing on Astronomical Data
**Target**: Stephan's Quintet (HCG 92) & Pillars of Creation (M16)
**Data Sources**: JWST NIRCam (F090W, F150W, F200W), Hubble HST

## Imports and Setup

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import math
import os
from astroquery.mast import Observations
from astropy.io import fits
from astropy.visualization import ImageNormalize, ZScaleInterval, AsinhStretch, LogStretch
from photutils.background import Background2D, MedianBackground

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## Environment Verification

In [ ]:
import astropy, astroquery, photutils, scipy

print(f"astropy:     {astropy.__version__}")
print(f"astroquery:  {astroquery.__version__}")
print(f"photutils:   {photutils.__version__}")
print(f"opencv:      {cv2.__version__}")
print(f"numpy:       {np.__version__}")
print(f"scipy:       {scipy.__version__}")

## Data Acquisition from MAST
Query and download FITS science products for two targets from Hubble and JWST archives.

In [ ]:
targets = {
    "Stephan's Quintet": "HCG 92",
    "Pillars of Creation": "M16"
}

def query_and_download(target_name, collection, filter_name=None):
    obs_table = Observations.query_object(target_name, radius="0.01 deg")
    filtered = obs_table[(obs_table['obs_collection'] == collection) &
                         (obs_table['dataproduct_type'] == 'image')]

    if filter_name and len(filtered) > 0:
        match = filtered[filtered['filters'] == filter_name]
        if len(match) > 0:
            filtered = match

    prods = Observations.get_product_list(filtered[0:1])
    sci = Observations.filter_products(prods, productType="SCIENCE", extension="fits")
    manifest = Observations.download_products(sci[0])

    obs_id = filtered[0]['obs_id'] if 'obs_id' in filtered.colnames else filtered[0]['obsid']
    proposal_id = filtered[0]['proposal_id'] if 'proposal_id' in filtered.colnames else "N/A"
    filt = filtered[0]['filters'] if 'filters' in filtered.colnames else filter_name

    print(f"  {collection} | {target_name} | Filter: {filt} | Obs ID: {obs_id} | Proposal: {proposal_id}")
    return manifest['Local Path'][0]

# JWST NIRCam channels for Stephan's Quintet
fits_sq_f090 = query_and_download(targets["Stephan's Quintet"], "JWST", "F090W")
fits_sq_f150 = query_and_download(targets["Stephan's Quintet"], "JWST", "F150W")
fits_sq_f200 = query_and_download(targets["Stephan's Quintet"], "JWST", "F200W")

# HST counterparts
fits_sq_hst = query_and_download(targets["Stephan's Quintet"], "HST")
fits_m16_hst = query_and_download(targets["Pillars of Creation"], "HST")

## FITS Header Inspection
Extract telescope, instrument, filter, exposure time, and array dimensions from each downloaded file.

In [ ]:
def inspect_header(path, label):
    with fits.open(path) as hdul:
        hdr = hdul[0].header
        sci_hdr = hdul[1].header if len(hdul) > 1 else hdr

        info = {
            "Telescope":  hdr.get('TELESCOP', sci_hdr.get('TELESCOP', '?')),
            "Instrument": hdr.get('INSTRUME', sci_hdr.get('INSTRUME', '?')),
            "Filter":     hdr.get('FILTER', sci_hdr.get('FILTER', hdr.get('FILTNAM1', '?'))),
            "Exp (s)":    hdr.get('EXPTIME', sci_hdr.get('EXPTIME', '?')),
            "Date":       hdr.get('DATE-OBS', sci_hdr.get('DATE-OBS', '?')),
        }

        for hdu in hdul:
            if hdu.data is not None and len(hdu.data.shape) >= 2:
                info["Shape"] = hdu.data.shape
                break

    print(f"[{label}]  {info}")
    return info

inspect_header(fits_sq_hst, "SQ Hubble")
inspect_header(fits_sq_f200, "SQ JWST F200W")
inspect_header(fits_m16_hst, "M16 Hubble")

## Preprocessing Pipeline
Background subtraction (`Background2D`), denoising (`fastNlMeansDenoising`), and Asinh stretch normalization applied to each filter channel.

In [ ]:
def load_fits_data(path):
    with fits.open(path) as hdul:
        for hdu in hdul:
            if hdu.data is not None and len(hdu.data.shape) >= 2:
                return hdu.data.astype(np.float32)
    return None

def process_channel(data, box_size=50, h=10):
    data = np.nan_to_num(data, nan=np.nanmedian(data))
    bkg = Background2D(data, box_size=box_size, filter_size=3, bkg_estimator=MedianBackground())
    data_sub = data - bkg.background

    d_min, d_max = np.percentile(data_sub, 1), np.percentile(data_sub, 99.5)
    data_clip = np.clip(data_sub, d_min, d_max)
    data_8 = ((data_clip - d_min) / (d_max - d_min + 1e-8) * 255).astype(np.uint8)

    denoised = cv2.fastNlMeansDenoising(data_8, h=h)

    norm = ImageNormalize(vmin=0, vmax=np.percentile(denoised, 99), stretch=AsinhStretch())
    out = norm(denoised)
    if np.ma.isMaskedArray(out):
        out = out.filled(0)
    return np.clip(out, 0, 1)

raw_f090 = load_fits_data(fits_sq_f090)
raw_f150 = load_fits_data(fits_sq_f150)
raw_f200 = load_fits_data(fits_sq_f200)
raw_sq_hst = load_fits_data(fits_sq_hst)
raw_m16_hst = load_fits_data(fits_m16_hst)

## Raw Filter Channel Preview

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("JWST NIRCam Raw Channels — Stephan's Quintet", fontweight="bold")

for ax, (title, data, cmap) in zip(axes, [
    ("F090W", raw_f090, "Blues_r"),
    ("F150W", raw_f150, "Greens_r"),
    ("F200W", raw_f200, "Reds_r")
]):
    vmin, vmax = np.percentile(np.nan_to_num(data), [1, 99])
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax, origin="lower")
    ax.set_title(f"{title}  ({data.shape})")
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout(); plt.show()

## Processed Channels and RGB Composite

In [ ]:
proc_f090 = process_channel(raw_f090)
proc_f150 = process_channel(raw_f150)
proc_f200 = process_channel(raw_f200)
proc_sq_hst = process_channel(raw_sq_hst)
proc_m16_hst = process_channel(raw_m16_hst)

min_h = min(proc_f090.shape[0], proc_f150.shape[0], proc_f200.shape[0])
min_w = min(proc_f090.shape[1], proc_f150.shape[1], proc_f200.shape[1])

r_ch = proc_f200[:min_h, :min_w]
g_ch = proc_f150[:min_h, :min_w]
b_ch = proc_f090[:min_h, :min_w]
jwst_rgb = np.dstack([r_ch, g_ch, b_ch])

plt.figure(figsize=(10, 10))
plt.imshow(jwst_rgb, origin="lower")
plt.title("Stephan's Quintet — JWST False-Color (R:F200W  G:F150W  B:F090W)", fontweight="bold")
plt.axis("off")
plt.savefig("stephans_quintet_jwst_rgb.png", bbox_inches='tight', dpi=150)
plt.show()

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(r_ch, cmap="Reds_r", origin="lower"); axes[0].set_title("F200W (Red)"); axes[0].axis("off")
axes[1].imshow(g_ch, cmap="Greens_r", origin="lower"); axes[1].set_title("F150W (Green)"); axes[1].axis("off")
axes[2].imshow(b_ch, cmap="Blues_r", origin="lower"); axes[2].set_title("F090W (Blue)"); axes[2].axis("off")
axes[3].imshow(jwst_rgb, origin="lower"); axes[3].set_title("Composite RGB"); axes[3].axis("off")
plt.tight_layout(); plt.show()

## Hubble vs JWST Side-by-Side

In [ ]:
def crop_center(img, size=1024):
    h, w = img.shape[:2]
    cy, cx = h // 2, w // 2
    half = size // 2
    return img[max(0, cy-half):min(h, cy+half), max(0, cx-half):min(w, cx+half)]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle("Stephan's Quintet — Hubble vs JWST", fontweight="bold")

axes[0].imshow(crop_center(proc_sq_hst), cmap="gray", origin="lower")
axes[0].set_title("Hubble (Visible/UV)"); axes[0].axis("off")

axes[1].imshow(crop_center(jwst_rgb), origin="lower")
axes[1].set_title("JWST (Near-Infrared Composite)"); axes[1].axis("off")

plt.tight_layout()
plt.savefig("comparison_hubble_vs_jwst.png", bbox_inches='tight', dpi=150)
plt.show()

## Pillars of Creation (M16) — Hubble

In [ ]:
plt.figure(figsize=(8, 8))
plt.imshow(crop_center(proc_m16_hst), cmap="inferno", origin="lower")
plt.title("M16 — Pillars of Creation (Hubble)", fontweight="bold")
plt.axis("off")
plt.savefig("pillars_of_creation_m16.png", bbox_inches='tight', dpi=150)
plt.show()

## Galaxy Zoo Morphological Mapping
Classification mapping for Stephan's Quintet members against the Galaxy Zoo taxonomy.

In [ ]:
members = [
    ("NGC 7317", "Elliptical (E4)", "Smooth", "Core group member"),
    ("NGC 7318A", "Interacting", "Merger", "Active collision"),
    ("NGC 7318B", "Interacting Spiral", "Merger / Tidal", "Shock wave"),
    ("NGC 7319", "Barred Spiral SB(s)bc", "Featured / Barred", "Tidal tails"),
    ("NGC 7320", "Dwarf Spiral", "Featured Spiral", "Foreground"),
]

print(f"{'Galaxy':<12}  {'Type':<24}  {'GZ Class':<20}  {'Notes'}")
print("-" * 78)
for name, morph, gz, note in members:
    print(f"{name:<12}  {morph:<24}  {gz:<20}  {note}")

## Test Image Extraction
Convert the JWST composite to grayscale and extract a 512×512 central patch for the DIP exercises below.

In [ ]:
img_gray = cv2.cvtColor((jwst_rgb * 255).astype(np.float32), cv2.COLOR_RGB2GRAY).astype(np.uint8)

h, w = img_gray.shape
cy, cx = h // 2, w // 2
img = img_gray[cy-256:cy+256, cx-256:cx+256]
H, W = img.shape

plt.figure(figsize=(5, 5))
plt.imshow(img, cmap="gray", origin="lower")
plt.title(f"Test Image ({H}x{W})")
plt.axis("off"); plt.show()

## Sampling / Downsampling
Downsample by 1/2, 1/4, 1/8 and upscale back with nearest-neighbor to visualize information loss.

In [ ]:
scales = [2, 4, 8]
downsampled = {}
for s in scales:
    downsampled[s] = img[::s, ::s]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle("Downsampling Comparison", fontweight="bold")

axes[0].imshow(img, cmap="gray", vmin=0, vmax=255, origin="lower")
axes[0].set_title("Original"); axes[0].axis("off")

for ax, s in zip(axes[1:], scales):
    small = downsampled[s]
    restored = cv2.resize(small, (W, H), interpolation=cv2.INTER_NEAREST)
    ax.imshow(restored, cmap="gray", vmin=0, vmax=255, origin="lower")
    ax.set_title(f"1/{s}  ({small.shape[0]}x{small.shape[1]})"); ax.axis("off")

plt.tight_layout(); plt.show()

## Interpolation Comparison
Upscale the 1/4 image (128×128) back to 512×512 using nearest-neighbor, bilinear, and bicubic.

In [ ]:
small = downsampled[4]

methods = {
    "Nearest": cv2.INTER_NEAREST,
    "Bilinear": cv2.INTER_LINEAR,
    "Bicubic": cv2.INTER_CUBIC,
}

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle("Interpolation Methods (128x128 -> 512x512)", fontweight="bold")

axes[0].imshow(img, cmap="gray", vmin=0, vmax=255, origin="lower")
axes[0].set_title("Original"); axes[0].axis("off")

for ax, (name, flag) in zip(axes[1:], methods.items()):
    up = cv2.resize(small, (W, H), interpolation=flag)
    ax.imshow(up, cmap="gray", vmin=0, vmax=255, origin="lower")
    ax.set_title(name); ax.axis("off")

plt.tight_layout(); plt.show()

## Aliasing / Stripe Experiment
A period-8 stripe pattern sampled at various strides to demonstrate sub-Nyquist aliasing.

In [ ]:
sw = 256
period = 8
stripe = np.zeros((sw, sw), dtype=np.uint8)
for c in range(sw):
    stripe[:, c] = 255 if (c % period) < (period // 2) else 0

samples = {
    "Original": stripe[::1, ::1],
    "Stride=4": stripe[::1, ::4],
    "Stride=7 (alias)": stripe[::1, ::7],
    "Stride=8 (miss)": stripe[::1, ::8],
}

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle("Aliasing Demo (Stripe Period = 8 px)", fontweight="bold")

for ax, (t, s) in zip(axes, samples.items()):
    d = cv2.resize(s, (sw, sw), interpolation=cv2.INTER_NEAREST)
    ax.imshow(d, cmap="gray", vmin=0, vmax=255, origin="lower")
    ax.set_title(t); ax.axis("off")

plt.tight_layout(); plt.show()

## Quantization
Reduce intensity precision from 256 levels down to 32, 8, 4, and 2.

In [ ]:
def quantize(im, levels):
    f = 256.0 / levels
    return ((im / f).astype(np.uint8) * f).astype(np.uint8)

lvls = [256, 32, 8, 4, 2]
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
fig.suptitle("Intensity Quantization", fontweight="bold")

for ax, lv in zip(axes, lvls):
    ax.imshow(quantize(img, lv), cmap="gray", vmin=0, vmax=255, origin="lower")
    ax.set_title(f"{lv} levels ({int(math.log2(lv))} bit)"); ax.axis("off")

plt.tight_layout(); plt.show()

## Pixel Neighborhoods
N4, ND, and N8 neighborhoods for an interior pixel.

In [ ]:
px, py = 5, 5
N4  = [(px-1, py), (px+1, py), (px, py-1), (px, py+1)]
ND  = [(px-1, py-1), (px-1, py+1), (px+1, py-1), (px+1, py+1)]
N8  = N4 + ND

print(f"P = ({px}, {py})")
print(f"N4:  {N4}")
print(f"ND:  {ND}")
print(f"N8:  {N8}")

patch = np.ones((11, 11, 3), dtype=np.uint8) * 235
c = 5
patch[c, c] = [50, 50, 220]
for r, cc in [(c-1,c),(c+1,c),(c,c-1),(c,c+1)]:
    patch[r, cc] = [50, 180, 50]
for r, cc in [(c-1,c-1),(c-1,c+1),(c+1,c-1),(c+1,c+1)]:
    patch[r, cc] = [220, 50, 50]

fig, ax = plt.subplots(figsize=(3.5, 3.5))
ax.imshow(patch, interpolation="nearest")
ax.set_title("Blue=P  Green=N4  Red=ND", fontsize=9)
for i in range(12):
    ax.axhline(i-0.5, color='w', lw=0.5)
    ax.axvline(i-0.5, color='w', lw=0.5)
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## Distance Measures
Euclidean, City-block, and Chessboard distances between two points.

In [ ]:
x1, y1 = 2, 3
x2, y2 = 7, 8
dx, dy = abs(x1-x2), abs(y1-y2)

D_E = math.sqrt(dx**2 + dy**2)
D4  = dx + dy
D8  = max(dx, dy)

print(f"P1=({x1},{y1})  P2=({x2},{y2})  dx={dx}  dy={dy}")
print(f"Euclidean:  sqrt({dx}^2+{dy}^2) = {D_E:.4f}")
print(f"City-block: {dx}+{dy} = {D4}")
print(f"Chessboard: max({dx},{dy}) = {D8}")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (t, d) in zip(axes, [("Euclidean", f"{D_E:.3f}"), ("City-block", D4), ("Chessboard", D8)]):
    g = np.ones((11, 11, 3), dtype=np.uint8) * 240
    g[x1, y1] = [50, 50, 220]; g[x2, y2] = [220, 50, 50]
    ax.imshow(g, interpolation="nearest")
    ax.set_title(f"{t} = {d}")
    ax.set_xticks(np.arange(-0.5, 11, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, 11, 1), minor=True)
    ax.grid(which="minor", color="grey", lw=0.5)
    ax.tick_params(which="both", bottom=False, left=False, labelbottom=False, labelleft=False)
plt.tight_layout(); plt.show()

## Affine Transformation
Rotation (30°) + translation (40, 20) with hand-calculated landmark verification.

In [ ]:
theta_deg = 30.0
tx, ty = 40.0, 20.0
theta = math.radians(theta_deg)
cos_t, sin_t = math.cos(theta), math.sin(theta)

M = np.array([[cos_t, -sin_t, tx],
              [sin_t,  cos_t, ty]], dtype=np.float64)

lx, ly = W // 2, H // 2
lx_new = cos_t * lx - sin_t * ly + tx
ly_new = sin_t * lx + cos_t * ly + ty

print(f"Landmark ({lx},{ly}) -> ({lx_new:.2f}, {ly_new:.2f})")

transformed = cv2.warpAffine(img, M, (W, H))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(f"Affine: rot={theta_deg} deg, t=({tx},{ty})", fontweight="bold")

axes[0].imshow(img, cmap="gray", vmin=0, vmax=255, origin="lower")
axes[0].plot(lx, ly, 'ro', ms=9); axes[0].set_title("Original"); axes[0].axis("off")

axes[1].imshow(transformed, cmap="gray", vmin=0, vmax=255, origin="lower")
axes[1].plot(lx_new, ly_new, 'ro', ms=9); axes[1].set_title("Transformed"); axes[1].axis("off")

plt.tight_layout(); plt.show()

## Affine + Scaling Sweep
Scale factor swept from 1.0 to 1.3 combined with the rotation and translation above.

In [ ]:
scale_factors = [1.0, 1.1, 1.2, 1.3]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle("Affine + Scaling (s = 1.0 to 1.3)", fontweight="bold")

for ax, s in zip(axes, scale_factors):
    M_s = np.array([[s*cos_t, -s*sin_t, tx],
                    [s*sin_t,  s*cos_t, ty]], dtype=np.float64)

    lx_s = s*cos_t*lx - s*sin_t*ly + tx
    ly_s = s*sin_t*lx + s*cos_t*ly + ty

    warped = cv2.warpAffine(img, M_s, (W, H))
    ax.imshow(warped, cmap="gray", vmin=0, vmax=255, origin="lower")
    ax.plot(lx_s, ly_s, 'ro', ms=7)
    ax.set_title(f"s={s:.1f}  ({lx_s:.0f},{ly_s:.0f})"); ax.axis("off")

plt.tight_layout(); plt.show()